In [ ]:
import random
import numpy as np
from tqdm import tqdm
import os

# Loading the graph dataset
def load_citeseer_graph(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"The specified path {path} does not exist.")
    
    # loading the graph from edge list format (I do it manually because it is banned to use networkx library)
    G = {}
    with open(path, 'r') as file:
        for line in file:
            node_a, node_b = line.strip().split()
            if node_a not in G:
                G[node_a] = []
            G[node_a].append(node_b)
            if node_b not in G:
                G[node_b] = []  # Ensuring that every node appears in the graph (even if it has no outgoing edges)
    return G

def influence_spread(G, seed_set, p, num_realizations):
    spread = 0
    for _ in range(num_realizations):
        activated = set(seed_set)
        new_activated = set(seed_set)
        while new_activated:
            next_activated = set()
            for node in new_activated:
                for neighbor in G.get(node, []):
                    if neighbor not in activated and random.random() < p:
                        next_activated.add(neighbor)
            activated |= next_activated
            new_activated = next_activated
        spread += len(activated)
    return spread / num_realizations

def hill_climbing(G, k, p, num_realizations):
    seed_set = set()
    for _ in range(k):
        best_node = None
        best_spread = -1
        for node in G:
            if node not in seed_set:
                spread = influence_spread(G, seed_set | {node}, p, num_realizations)
                if spread > best_spread:
                    best_spread = spread
                    best_node = node
        seed_set.add(best_node)
    return list(seed_set)

def optimized_hill_climbing(G, k, p, num_realizations):
    seed_set = set()
    node_spreads = {node: influence_spread(G, {node}, p, num_realizations) for node in G}
    for _ in range(k):
        best_node = max(node_spreads, key=node_spreads.get)
        seed_set.add(best_node)
        del node_spreads[best_node]
        for node in node_spreads:
            node_spreads[node] = influence_spread(G, seed_set | {node}, p, num_realizations)
    return list(seed_set)

# Experimenting with different number of realizations and probabilities
def analyze_stability(G, k, num_realizations_list, p_list):
    results = {}
    for p in p_list:
        for num_realizations in num_realizations_list:
            print(f"Running analysis for p={p}, num_realizations={num_realizations}")
            seed_sets = []
            for _ in range(5):   # Runing multiple (5) times to check consistency
                seed_set = optimized_hill_climbing(G, k, p, num_realizations)
                seed_sets.append(seed_set)
            # Finding nodes that consistently appear in the top k
            consistent_nodes = set(seed_sets[0])
            for seed_set in seed_sets[1:]:
                consistent_nodes &= set(seed_set)
            results[(p, num_realizations)] = consistent_nodes
    return results

def main():
    path = "F:/Aidin/Universities/Amirkabir (Master)/Lessons/Term 2/Analysis of Complex Networks/Assignments/HW1/Answer/input/P3/citeseer-cites.txt"
    G = load_citeseer_graph(path)
    k = 20  # Number of nodes to find
    num_realizations_list = [20, 100, 500]
    p_list = [0.1, 0.2, 0.5, 0.8]

    # Runing basic hill climbing
    print("Running Basic Hill Climbing...")
    top_20_basic = hill_climbing(G, k, p=0.1, num_realizations=100)
    print("Top 20 nodes (Basic Hill Climbing):", top_20_basic)

    # Runing optimized hill climbing
    print("Running Optimized Hill Climbing...")
    top_20_optimized = optimized_hill_climbing(G, k, p=0.1, num_realizations=100)
    print("Top 20 nodes (Optimized Hill Climbing):", top_20_optimized)

    # Analyzing stability
    stability_results = analyze_stability(G, k, num_realizations_list, p_list)
    for key, consistent_nodes in stability_results.items():
        p, num_realizations = key
        print(f"Consistent nodes for p={p}, num_realizations={num_realizations}: {consistent_nodes}")

if __name__ == "__main__":
    main()


Running Basic Hill Climbing...
Top 20 nodes (Basic Hill Climbing): ['grolimund95integrating', 'decker95environment', 'brin98anatomy', 'bharat98improved', '21158', 'lawrence98searching', '87928', 'jantke93casebased', 'craven98learning', 'carson99blobworld', 'rao95bdi', 'jennings95controlling', 'chakrabarti98automatic', 'kinny96methodology', '48055', '282608', 'dreilinger97experiences', 'florescu99query', 'kitano97robocup', '32521']
Running Optimized Hill Climbing...
Top 20 nodes (Optimized Hill Climbing): ['iglesias99survey', 'decker95environment', 'brin98anatomy', 'bharat98improved', 'soderland99learning', 'pavlovic97visual', 'lawrence98searching', '87928', 'craven98learning', 'rao95bdi', 'jennings95controlling', '15156', 'starner98realtime', 'aha91casebased', 'chakrabarti98automatic', '254597', 'dreilinger97experiences', 'stockinger01design', 'asada98robocup', '32521']
Running analysis for p=0.1, num_realizations=20
Running analysis for p=0.1, num_realizations=100
Running analysis for